# 07 — Annexe : fusionner l'adapter, exporter en GGUF, utiliser llama.cpp

**Chapitre 3** · annexe / devoir · GPU recommandé · [![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zenocode-org/ai-masterclass/blob/main/chapitre_3/07_annexe_fusion_gguf.ipynb)

![Parcours du chapitre 3](https://raw.githubusercontent.com/zenocode-org/ai-masterclass/main/chapitre_3/figures/00_parcours.png)

À la fin du notebook 05, on a **deux** artefacts : le modèle de base (14,5 Go, sur Hugging Face) et notre **adapter** (~160 Mo). Pour déployer le modèle hors de Python (sur un portable, dans une application), on utilise souvent **llama.cpp** et son format de fichier **GGUF**. Deux chemins :

| Chemin | Ce qu'on produit | Ressources | Intérêt |
|---|---|---|---|
| **A. Adapter → GGUF** | un petit fichier GGUF d'adapter, appliqué à une base GGUF existante | légères (CPU, quelques minutes) | garder la modularité : une base, plusieurs adapters |
| **B. Fusion → GGUF** | un seul modèle GGUF, adapter intégré | ~16 Go de RAM ou un GPU ≥ 24 Go, ~30 Go de disque | un fichier unique, aucun surcoût à l'inférence |

> **Statut :** la fusion (§1) a été testée sur un petit modèle de même architecture. Les commandes llama.cpp (§2 et §3) suivent la documentation du projet mais n'ont pas été exécutées de bout en bout sur le 7B : vérifiez-les avant le cours, llama.cpp évolue vite.

In [ ]:
# 0. Installation (Colab) — versions testées ensemble ; ~1 min
import os, sys
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    %pip install -q "transformers==5.22.1" "trl==1.17.1" "peft==0.21.2" "bitsandbytes==0.50.2" "accelerate==1.15.0" "datasets==5.1.0"
    if not os.path.exists("finetune.py"):
        !git clone -q --depth 1 https://github.com/zenocode-org/ai-masterclass
        %cd ai-masterclass/chapitre_3

import math, json, time, random, warnings
warnings.filterwarnings("ignore", message="IProgress not found")
import torch
import matplotlib.pyplot as plt
import finetune as ft

assert torch.cuda.is_available(), "Pas de GPU : sur Colab, Exécution > Modifier le type d'exécution > GPU (T4)."
GPU = torch.cuda.get_device_name(0)
MEMOIRE_GPU = torch.cuda.get_device_properties(0).total_memory / 1024**3
BF16 = torch.cuda.get_device_capability(0)[0] >= 8   # Ampere (A100, L4…) et plus récents ; pas le T4
DTYPE = torch.bfloat16 if BF16 else torch.float16
print(f"GPU : {GPU} ({MEMOIRE_GPU:.1f} Go) | calcul en {'BF16' if BF16 else 'FP16 (pas de BF16 sur ce GPU)'}")
print("PyTorch", torch.__version__)

In [ ]:
CORPUS = "kaamelott"
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"
DOSSIER_ADAPTER = f"adapters/mistral7b-{CORPUS}-qlora"
FAIRE_FUSION = True            # §1 : demande ~16 Go de RAM (Colab « RAM élevée ») ou un GPU >= 24 Go
FAIRE_GGUF = True              # §2 : clone llama.cpp, convertit l'adapter, télécharge une base GGUF (4,4 Go)

from pathlib import Path
assert Path(DOSSIER_ADAPTER).exists(), "Lancez d'abord le notebook 05 (ou recopiez l'adapter depuis Google Drive)."

## 1. Fusionner l'adapter dans le modèle (`merge_and_unload`)

C'est la fusion du notebook 03, §5, appliquée à 7 milliards de paramètres : chaque couche ciblée devient `W₀ + (α/r)·BA`. **Important :** on fusionne dans une base en **16 bits**, pas dans la base 4 bits de l'entraînement : sinon on écrirait `ΔW` dans une matrice quantifiée et on perdrait l'essentiel de la correction.

In [ ]:
if FAIRE_FUSION:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel

    sur_gpu = MEMOIRE_GPU >= 20                   # le modèle FP16 (14,5 Go) tient sur un L4 / A100, pas sur un T4
    print("fusion sur", "GPU" if sur_gpu else "CPU (RAM)")
    base = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16,
                                                device_map={"": 0} if sur_gpu else "cpu", low_cpu_mem_usage=True)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = PeftModel.from_pretrained(base, DOSSIER_ADAPTER)

    # vérification : la fusion ne change pas les sorties (aux arrondis FP16 près)
    test = tokenizer("ARTHUR: Bon, on attaque ?", return_tensors="pt").to(base.device)
    with torch.no_grad():
        avant = model(**test).logits.float()
    fusionne = model.merge_and_unload()
    with torch.no_grad():
        apres = fusionne(**test).logits.float()
    print(f"écart max des logits avant/après fusion : {(avant - apres).abs().max().item():.3f} "
          f"(ordre de grandeur des logits : {avant.abs().max().item():.1f})")
    print("reste-t-il des couches LoRA ?", any("lora" in n for n, _ in fusionne.named_modules()))

    DOSSIER_FUSION = f"merged/mistral7b-{CORPUS}"
    fusionne.save_pretrained(DOSSIER_FUSION, max_shard_size="2GB")
    tokenizer.save_pretrained(DOSSIER_FUSION)
    taille = sum(f.stat().st_size for f in Path(DOSSIER_FUSION).iterdir()) / 1024**3
    print(f"modèle fusionné : {DOSSIER_FUSION} ({taille:.1f} Go)")
    del base, model, fusionne
    torch.cuda.empty_cache()

Le dossier `merged/...` est un modèle Hugging Face ordinaire : `AutoModelForCausalLM.from_pretrained(DOSSIER_FUSION)` le charge sans `peft`. On a perdu la modularité (un modèle par style) mais gagné la simplicité.

## 2. Chemin A : l'adapter en GGUF, appliqué à une base GGUF

1. `convert_lora_to_gguf.py` (llama.cpp) lit `adapter_config.json`, retrouve la configuration du modèle de base sur Hugging Face, et écrit l'adapter au format GGUF ;
2. on télécharge une base **déjà quantifiée** en GGUF (Q4_K_M, 4,4 Go) ;
3. `llama-cpp-python` charge la base et applique l'adapter au moment de l'inférence.

Remarque : l'adapter a été entraîné sur une base NF4 et sera appliqué à une base Q4_K_M. Les deux quantifications diffèrent un peu ; en pratique l'effet est faible, mais c'est une source d'écart à mesurer.

In [ ]:
if FAIRE_GGUF:
    if not Path("llama.cpp").exists():
        !git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
    %pip install -q -e llama.cpp/gguf-py
    !python llama.cpp/convert_lora_to_gguf.py {DOSSIER_ADAPTER} --outtype f16 --outfile adapters/{CORPUS}-lora-f16.gguf
    !ls -lh adapters/*.gguf

In [ ]:
if FAIRE_GGUF:
    from huggingface_hub import hf_hub_download
    base_gguf = hf_hub_download("bartowski/Mistral-7B-Instruct-v0.3-GGUF", "Mistral-7B-Instruct-v0.3-Q4_K_M.gguf")
    print(base_gguf)
    # llama-cpp-python se compile à l'installation : ~5 min en version CPU.
    # Version GPU (plus longue à compiler) : CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python
    %pip install -q llama-cpp-python

In [ ]:
if FAIRE_GGUF:
    from llama_cpp import Llama
    prompt = [{"role": "user", "content": f"{ft.CONSIGNES[CORPUS]}\n\nARTHUR: Bon. Quelqu’un a une idée pour récupérer le Graal avant ce soir ?"}]
    for lora in [None, f"adapters/{CORPUS}-lora-f16.gguf"]:
        llm = Llama(model_path=base_gguf, lora_path=lora, n_ctx=1024, n_gpu_layers=0, seed=0, verbose=False)
        sortie = llm.create_chat_completion(messages=prompt, max_tokens=80, temperature=0.7)
        print("avec adapter" if lora else "base seule  ", ":", sortie["choices"][0]["message"]["content"])
        del llm

En ligne de commande, avec les binaires de llama.cpp, c'est l'option `--lora` :

```bash
llama-cli -m Mistral-7B-Instruct-v0.3-Q4_K_M.gguf --lora kaamelott-lora-f16.gguf -cnv
```

## 3. Chemin B : le modèle fusionné en GGUF (pour aller plus loin)

À partir du dossier `merged/...` du §1 (≈ 30 Go de disque au total) :

```bash
# 1. conversion Hugging Face -> GGUF 16 bits (~14,5 Go)
python llama.cpp/convert_hf_to_gguf.py merged/mistral7b-kaamelott --outtype f16 --outfile kaamelott-f16.gguf
# 2. compilation de llama.cpp (une fois), puis quantification en 4 bits (~4,4 Go)
cmake -B llama.cpp/build llama.cpp && cmake --build llama.cpp/build --target llama-quantize llama-cli -j
llama.cpp/build/bin/llama-quantize kaamelott-f16.gguf kaamelott-Q4_K_M.gguf Q4_K_M
# 3. utilisation
llama.cpp/build/bin/llama-cli -m kaamelott-Q4_K_M.gguf -cnv
```

Le fichier `kaamelott-Q4_K_M.gguf` fonctionne dans tous les outils compatibles GGUF (llama.cpp, Ollama, LM Studio…), sur un portable sans GPU.

## Licence et redistribution

Avant de **publier** un adapter ou un modèle fusionné, distinguez :
- la licence des **poids Mistral** : Apache 2.0, redistribution autorisée ;
- les droits sur les **données** : les dialogues de *Kaamelott* sont une œuvre protégée ; une transcription disponible en ligne n'est pas libre de droits. Un adapter qui **mémorise** des répliques (notebook 06) redistribue une partie de l'œuvre. Platon (traduction Cousin, XIXᵉ s.) et Shakespeare sont dans le domaine public.

« Téléchargeable » ne veut pas dire « librement redistribuable ».

## À retenir

- `merge_and_unload()` écrit `W₀ + (α/r)·BA` dans chaque couche : **même modèle**, plus de dépendance à `peft`, aucun surcoût à l'inférence. Toujours fusionner dans une base **16 bits**.
- **GGUF** est le format de llama.cpp ; on peut convertir **l'adapter seul** (chemin léger, modulaire) ou le **modèle fusionné** puis le quantifier (fichier unique).
- Publier un modèle fine-tuné engage les droits sur les **données**, pas seulement la licence du modèle de base.